# x402 Two-Sided Demo — Notebook Version

This notebook breaks the original FastAPI provider and Streamlit dashboard into step-by-step Jupyter cells for VS Code.

The workflow is:

1. Load the existing `.env` in this folder values.
2. Write/start the x402-protected third-party provider server.
3. Inspect the provider.
4. Call the protected endpoint without payment and observe HTTP 402.
5. Authorize the buyer-side agent payment.
6. Call the protected endpoint with x402 payment.
7. Inspect the service response and testnet evidence.

> Keep the provider running in a separate VS Code terminal while you execute the client-side notebook cells.

## 0. Install dependencies

Run this cell if your VS Code Python environment does not already have the required packages.

In [1]:
%pip install fastapi uvicorn python-dotenv requests eth-account x402

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Confirm the existing `.env` file

This notebook assumes a `.env` file already exists in the same folder as the notebook/script files and contains:

```env
PRIVATE_KEY=0x...
TPP_RECEIVE_ADDRESS=0x...
```

The buyer wallet needs test USDC on Base Sepolia.

In [2]:
from pathlib import Path

env_path = Path('.env')
if env_path.exists():
    print(f'Found existing .env at: {env_path.resolve()}')
else:
    raise FileNotFoundError(
        'No .env file found in the current working directory. '
        'Open this notebook from the folder that already contains your .env file, '
        'or change the working directory before running the remaining cells.'
    )

Found existing .env at: C:\Users\JumpStart\GitHub\Prototyp-Machine-Customer\code\.env


## 2. Provider/server code

This writes the FastAPI x402-protected TPP server to `x402_tpp_server.py`. Start it from a separate VS Code terminal in the next step.

In [3]:
%%writefile x402_tpp_server.py
"""
x402_tpp_server.py

Local third-party provider / TPP for the x402 demo.

Run in Terminal 1:
    python -m uvicorn x402_tpp_server:app --host 127.0.0.1 --port 4021

.env in same folder:
    TPP_RECEIVE_ADDRESS=0xYOUR_TPP_RECEIVING_ADDRESS

The provider exposes:
    GET /
    GET /api/v1/dummy-service

The dummy service is protected by x402. Without payment it returns HTTP 402.
With a valid x402 payment it returns a dummy JSON service response.
"""

import os
from typing import Any

from dotenv import load_dotenv
from fastapi import FastAPI

from x402.http import FacilitatorConfig, HTTPFacilitatorClient, PaymentOption
from x402.http.middleware.fastapi import PaymentMiddlewareASGI
from x402.http.types import RouteConfig
from x402.mechanisms.evm.exact import ExactEvmServerScheme
from x402.schemas import Network
from x402.server import x402ResourceServer


load_dotenv()

TPP_RECEIVE_ADDRESS = os.getenv("TPP_RECEIVE_ADDRESS")

HOST = "127.0.0.1"
PORT = 4021
PROTECTED_PATH = "/api/v1/dummy-service"

# Base Sepolia testnet. Base mainnet would be eip155:8453.
EVM_NETWORK: Network = "eip155:84532"

# Public x402 test facilitator base URL.
FACILITATOR_URL = "https://x402.org/facilitator"

# Demo price.
PRICE = "$0.001"


if not TPP_RECEIVE_ADDRESS:
    raise RuntimeError(
        "Missing TPP_RECEIVE_ADDRESS in .env. "
        "Example: TPP_RECEIVE_ADDRESS=0xYOUR_TPP_RECEIVING_ADDRESS"
    )

if not TPP_RECEIVE_ADDRESS.startswith("0x"):
    raise RuntimeError("TPP_RECEIVE_ADDRESS must be an EVM address starting with 0x.")


def create_app() -> FastAPI:
    provider_app = FastAPI(
        title="x402 Demo Third-Party Provider",
        description="Dummy x402-protected third-party provider.",
        version="1.0.0",
    )

    facilitator = HTTPFacilitatorClient(
        FacilitatorConfig(url=FACILITATOR_URL)
    )

    resource_server = x402ResourceServer(facilitator)
    resource_server.register(EVM_NETWORK, ExactEvmServerScheme())

    routes: dict[str, RouteConfig] = {
        f"GET {PROTECTED_PATH}": RouteConfig(
            accepts=[
                PaymentOption(
                    scheme="exact",
                    pay_to=TPP_RECEIVE_ADDRESS,
                    price=PRICE,
                    network=EVM_NETWORK,
                )
            ],
            mime_type="application/json",
            description="Paid dummy API service for x402 demonstration.",
        )
    }

    provider_app.add_middleware(
        PaymentMiddlewareASGI,
        routes=routes,
        server=resource_server,
    )

    @provider_app.get("/")
    async def root() -> dict[str, Any]:
        return {
            "role": "third-party provider / TPP",
            "service": "dummy paid API",
            "protected_endpoint": PROTECTED_PATH,
            "price": PRICE,
            "network": EVM_NETWORK,
            "pay_to": TPP_RECEIVE_ADDRESS,
            "facilitator": FACILITATOR_URL,
            "demo_note": (
                "This is the provider side. The protected route returns "
                "HTTP 402 unless paid via x402."
            ),
        }

    @provider_app.get(PROTECTED_PATH)
    async def dummy_service() -> dict[str, Any]:
        return {
            "service_received": True,
            "service_name": "Dummy Third-Party API",
            "message": (
                "The buyer agent has paid through x402. "
                "This is the protected service response."
            ),
            "payload": {
                "example_result": "42",
                "note": (
                    "This endpoint intentionally leads nowhere beyond this "
                    "dummy response."
                ),
            },
        }

    return provider_app


# This global variable is required by:
# python -m uvicorn x402_tpp_server:app --host 127.0.0.1 --port 4021
app = create_app()

Overwriting x402_tpp_server.py


## 3. Start the provider server

Run this command in a separate VS Code terminal, not inside the notebook:

```powershell
python -m uvicorn x402_tpp_server:app --host 127.0.0.1 --port 4021
```

Leave that terminal running, then continue below.

## 4. Client-side imports and configuration

In [4]:
import base64
import json
import os
import re
from pprint import pprint
from typing import Any

import requests
from dotenv import load_dotenv
from eth_account import Account

from x402 import x402ClientSync
from x402.http import x402HTTPClientSync
from x402.http.clients import x402_requests
from x402.mechanisms.evm import EthAccountSigner
from x402.mechanisms.evm.exact.register import register_exact_evm_client

load_dotenv()

PRIVATE_KEY = os.getenv("PRIVATE_KEY")
TPP_RECEIVE_ADDRESS = os.getenv("TPP_RECEIVE_ADDRESS")

HOST = "127.0.0.1"
PORT = 4021
PROTECTED_PATH = "/api/v1/dummy-service"

TPP_ROOT_URL = f"http://{HOST}:{PORT}/"
TPP_URL = f"http://{HOST}:{PORT}{PROTECTED_PATH}"

EVM_NETWORK = "eip155:84532"
PRICE = "$0.001"

BASE_SEPOLIA_TX_EXPLORER = "https://sepolia.basescan.org/tx/"
BASE_SEPOLIA_ADDRESS_EXPLORER = "https://sepolia.basescan.org/address/"

print("TPP root:", TPP_ROOT_URL)
print("Protected endpoint:", TPP_URL)
print("Network:", EVM_NETWORK)
print("Price:", PRICE)

TPP root: http://127.0.0.1:4021/
Protected endpoint: http://127.0.0.1:4021/api/v1/dummy-service
Network: eip155:84532
Price: $0.001


## 5. Helper functions

These are the non-UI helper functions from the original dashboard, adapted for notebook output.

In [5]:
def decode_possible_base64_json(value: str) -> Any:
    if not value:
        return None

    try:
        padded = value + "=" * (-len(value) % 4)
        raw = base64.b64decode(padded)
        return json.loads(raw.decode("utf-8"))
    except Exception:
        return value


def get_payment_headers(response: requests.Response) -> dict[str, Any]:
    result = {}

    for key, value in response.headers.items():
        if "payment" in key.lower() or "x402" in key.lower():
            result[key] = decode_possible_base64_json(value)

    return result


def response_body(response: requests.Response) -> Any:
    try:
        return response.json()
    except Exception:
        return response.text


def find_transaction_hash(obj: Any) -> str | None:
    tx_pattern = re.compile(r"^0x[a-fA-F0-9]{64}$")

    if obj is None:
        return None

    if isinstance(obj, str):
        value = obj.strip()
        if tx_pattern.match(value):
            return value
        return None

    # Support Pydantic v2 models
    if hasattr(obj, "model_dump"):
        try:
            return find_transaction_hash(obj.model_dump())
        except Exception:
            pass

    # Support Pydantic v1 models
    if hasattr(obj, "dict"):
        try:
            return find_transaction_hash(obj.dict())
        except Exception:
            pass

    # Support ordinary Python objects
    if hasattr(obj, "__dict__"):
        try:
            return find_transaction_hash(vars(obj))
        except Exception:
            pass

    if isinstance(obj, dict):
        preferred_keys = [
            "transaction",
            "txHash",
            "transactionHash",
            "transaction_hash",
            "settlementTxHash",
            "settlementTransactionHash",
            "hash",
        ]

        for key in preferred_keys:
            value = obj.get(key)
            found = find_transaction_hash(value)
            if found:
                return found

        for value in obj.values():
            found = find_transaction_hash(value)
            if found:
                return found

    if isinstance(obj, list):
        for item in obj:
            found = find_transaction_hash(item)
            if found:
                return found

    return None


def short_address(address: str) -> str:
    if not address or len(address) < 12:
        return address
    return f"{address[:6]}...{address[-4:]}"

## 6. Validate environment and wallet

In [6]:
def require_env() -> tuple[bool, str | None, Account | None]:
    if not PRIVATE_KEY:
        return False, "Missing PRIVATE_KEY in .env", None

    if not TPP_RECEIVE_ADDRESS:
        return False, "Missing TPP_RECEIVE_ADDRESS in .env", None

    try:
        buyer = Account.from_key(PRIVATE_KEY)
    except Exception as exc:
        return False, f"Invalid PRIVATE_KEY: {exc}", None

    if not TPP_RECEIVE_ADDRESS.startswith("0x"):
        return False, "TPP_RECEIVE_ADDRESS must start with 0x", None

    return True, None, buyer


ok, error, buyer = require_env()

if not ok:
    raise RuntimeError(error)

print("Buyer wallet:", buyer.address)
print("TPP receiving wallet:", TPP_RECEIVE_ADDRESS)
print("Buyer BaseScan:", BASE_SEPOLIA_ADDRESS_EXPLORER + buyer.address)
print("TPP receiver BaseScan:", BASE_SEPOLIA_ADDRESS_EXPLORER + TPP_RECEIVE_ADDRESS)

Buyer wallet: 0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81
TPP receiving wallet: 0xa0932E63446d7BC9C127dBeFD50BCF0E7Fa02B14
Buyer BaseScan: https://sepolia.basescan.org/address/0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81
TPP receiver BaseScan: https://sepolia.basescan.org/address/0xa0932E63446d7BC9C127dBeFD50BCF0E7Fa02B14


## 7. Inspect the provider

Expected result: HTTP 200 and a JSON body describing the protected endpoint.

In [9]:
def inspect_provider() -> dict[str, Any]:
    response = requests.get(TPP_ROOT_URL, timeout=20)

    return {
        "status_code": response.status_code,
        "body": response_body(response),
    }


provider_status = inspect_provider()
pprint(provider_status)

{'body': {'demo_note': 'This is the provider side. The protected route returns '
                       'HTTP 402 unless paid via x402.',
          'facilitator': 'https://x402.org/facilitator',
          'network': 'eip155:84532',
          'pay_to': '0xa0932E63446d7BC9C127dBeFD50BCF0E7Fa02B14',
          'price': '$0.001',
          'protected_endpoint': '/api/v1/dummy-service',
          'role': 'third-party provider / TPP',
          'service': 'dummy paid API'},
 'status_code': 200}


## 8. Call the protected API without payment

Expected result: HTTP 402 Payment Required, plus payment challenge headers.

In [10]:
def call_without_payment() -> dict[str, Any]:
    response = requests.get(TPP_URL, timeout=30)

    return {
        "status_code": response.status_code,
        "headers": get_payment_headers(response),
        "body": response_body(response),
    }


unpaid_result = call_without_payment()
pprint(unpaid_result)

if unpaid_result["status_code"] == 402:
    print("Success: provider correctly returned HTTP 402 Payment Required.")
else:
    print("Unexpected status code:", unpaid_result["status_code"])

{'body': {},
 'headers': {'payment-required': {'accepts': [{'amount': '1000',
                                               'asset': '0x036CbD53842c5426634e7929541eC2318f3dCF7e',
                                               'extra': {'name': 'USDC',
                                                         'version': '2'},
                                               'maxTimeoutSeconds': 300,
                                               'network': 'eip155:84532',
                                               'payTo': '0xa0932E63446d7BC9C127dBeFD50BCF0E7Fa02B14',
                                               'scheme': 'exact'}],
                                  'error': 'Payment required',
                                  'resource': {'description': 'Paid dummy API '
                                                              'service for '
                                                              'x402 '
                                                              'dem

## 9. Authorize and perform the x402 payment

This cell signs the payment payload with the buyer wallet and retries the protected API request.

In [11]:
def call_with_payment(buyer: Account) -> dict[str, Any]:
    x402_client = x402ClientSync()
    signer = EthAccountSigner(buyer)

    register_exact_evm_client(x402_client, signer)

    http_client = x402HTTPClientSync(x402_client)

    with x402_requests(x402_client) as session:
        response = session.get(TPP_URL, timeout=90)

    payment_headers = get_payment_headers(response)

    try:
        parsed_payment_response = http_client.get_payment_settle_response(
            lambda name: response.headers.get(name)
        )
    except Exception:
        parsed_payment_response = payment_headers

    tx_hash = find_transaction_hash(parsed_payment_response)

    if not tx_hash:
        tx_hash = find_transaction_hash(dict(response.headers))

    if not tx_hash:
        tx_hash = find_transaction_hash(response_body(response))

    return {
        "status_code": response.status_code,
        "headers": payment_headers,
        "body": response_body(response),
        "payment_response": parsed_payment_response,
        "transaction_hash": tx_hash,
    }


paid_result = call_with_payment(buyer)
pprint(paid_result)

if paid_result["status_code"] == 200:
    print("Success: payment accepted and protected service response returned.")
else:
    print("Paid request failed with HTTP status:", paid_result["status_code"])

{'body': {'message': 'The buyer agent has paid through x402. This is the '
                     'protected service response.',
          'payload': {'example_result': '42',
                      'note': 'This endpoint intentionally leads nowhere '
                              'beyond this dummy response.'},
          'service_name': 'Dummy Third-Party API',
          'service_received': True},
 'headers': {'payment-response': {'network': 'eip155:84532',
                                  'payer': '0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81',
                                  'success': True,
                                  'transaction': '0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f'}},
 'payment_response': SettleResponse(success=True, error_reason=None, error_message=None, payer='0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81', transaction='0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f', network='eip155:84532', amount=None),
 'status_code': 200

## 10. Inspect the service response

In [14]:
print("HTTP status:", paid_result.get("status_code"))

print("\nService response:")
pprint(paid_result.get("body"))

print("\nPayment response / settlement metadata:")
pprint(paid_result.get("payment_response"))

print("\nPayment-related headers:")
pprint(paid_result.get("headers"))

HTTP status: 200

Service response:
{'message': 'The buyer agent has paid through x402. This is the protected '
            'service response.',
 'payload': {'example_result': '42',
             'note': 'This endpoint intentionally leads nowhere beyond this '
                     'dummy response.'},
 'service_name': 'Dummy Third-Party API',
 'service_received': True}

Payment response / settlement metadata:
SettleResponse(success=True, error_reason=None, error_message=None, payer='0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81', transaction='0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f', network='eip155:84532', amount=None)

Payment-related headers:
{'payment-response': {'network': 'eip155:84532',
                      'payer': '0xe434ce93D0c05c22e3ff3eec406c9F18DBF3CB81',
                      'success': True,
                      'transaction': '0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f'}}


## 11. Testnet evidence

If the SDK or facilitator exposes a transaction hash, this cell prints the Base Sepolia BaseScan transaction URL. Otherwise, inspect the buyer and receiver addresses.

In [15]:
tx_hash = paid_result.get("transaction_hash")

if tx_hash:
    print("Transaction hash:", tx_hash)
    print("BaseScan transaction:", BASE_SEPOLIA_TX_EXPLORER + tx_hash)
else:
    print("No transaction hash exposed by the SDK/facilitator response.")
    print("Buyer address:", BASE_SEPOLIA_ADDRESS_EXPLORER + buyer.address)
    print("TPP receiver address:", BASE_SEPOLIA_ADDRESS_EXPLORER + TPP_RECEIVE_ADDRESS)

Transaction hash: 0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f
BaseScan transaction: https://sepolia.basescan.org/tx/0x91818a4470b634ba1557d5e25eb07c5f657243e5a8bb930ebcf87315e6c1080f


## 12. Demo narrative

1. The TPP exposes a normal API endpoint, but wraps it with x402 payment middleware.
2. A first unpaid request receives HTTP 402 Payment Required.
3. The presenter authorizes the buyer agent to proceed.
4. The buyer agent signs the payment payload with its wallet.
5. The paid request is retried.
6. The TPP releases the protected dummy service response.
7. The payment can be inspected on Base Sepolia, either by transaction hash or by wallet address.